# Data exploration — FLAIR aerial tiles + OCS GE labels

First pass over the data this pipeline will consume: one area (`D004_2021`),
a few tile zones. Run this notebook with the `Python (rsss)` kernel (the repo's
`uv` venv, already available in VS Code's kernel picker; see `src/rsss/data/`).

In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from rsss.data.data_loader import discover_data
from rsss.data.data_module import FlairDatamodule

# Repo/data root: walk up from wherever the kernel was started.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), f"data/ not found under {ROOT}"

DATA_ROOT = ROOT / "data"

SEED = 2026
NUM_CLASSES = 13  # FLAIR 13-class nomenclature, encoded 1..13 in MSK
AREA = "D004_2021"  # one small area first
TRAIN_ZONES = ["Z10_AU", "Z17_AA", "Z1_NN"]
VAL_ZONES = ["Z16_UU"]  # kept out of train for this notebook

random.seed(SEED)

# Band order of FLAIR-1 aerial rasters (official dataset card).
BAND_NAMES = ["Red", "Green", "Blue", "NIR", "nDSM"]

# FLAIR 13-class nomenclature (official IGN dataset card), MSK values 1..13.
FLAIR_CLASSES = {
    1: "Building",
    2: "Pervious surface",
    3: "Impervious surface",
    4: "Bare soil",
    5: "Water",
    6: "Coniferous",
    7: "Deciduous",
    8: "Brushwood",
    9: "Vineyard",
    10: "Herbaceous vegetation",
    11: "Agricultural land",
    12: "Plowed land",
    13: "Swimming pool",
}

CLASS_COLORS = [
    "#8c6d31",
    "#b3b3b3",
    "#4d4d4d",
    "#a0783c",
    "#0000d3",
    "#008b45",
    "#006400",
    "#9be303",
    "#c1292a",
    "#00ffe1",
    "#fafe78",
    "#86c9ca",
    "#ff01ff",
]
NUM_COLORS = NUM_CLASSES

print(f"ROOT      = {ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"Torch     = {torch.__version__}, rasterio = {rasterio.__version__}")

## 1. Inventory

`discover_data()` expects `<data_dir>/<area>/{aerial,labels}/<tile>/` and walks
it recursively. Note it pairs the resulting IMG/MSK lists **positionally**: a
zone where some image has no mask counterpart would silently desynchronize
everything after it. We check here that lengths and IDs line up.

In [ ]:
dict_all = discover_data(str(DATA_ROOT))
imgs, msks = dict_all["IMG"], dict_all["MSK"]
print(f"Discovered: {len(imgs)} images, {len(msks)} masks")

areas = Counter(Path(p).parts[-4] for p in imgs)
for area, n in sorted(areas.items()):
    n_msk = sum(1 for p in msks if Path(p).parts[-4] == area)
    print(f"  {area}: {n} images / {n_msk} masks")

same_ids = all(
    Path(i).stem.replace("IMG", "MSK") == Path(m).stem for i, m in zip(imgs, msks)
)
print(f"Positional IMG/MSK pairing by stem: {'OK' if same_ids else 'MISMATCH'}")

## 2. Pipeline specs (`discover_data` → `FlairDatamodule`)

`discover_data()` only scans whole areas, so to train on a few zones we
discover at the root and filter paths ourselves, then pass explicit
`dict_train` / `dict_val` to the datamodule (its fallback, when dicts are
omitted, assigns the *entire* discovered set to both train and val).

In [ ]:
def zones_dict(imgs: list[str], msks: list[str], zone_suffixes: list[str]) -> dict:
    """Join the IMG/MSK lists restricted to the given zones of AREA."""
    keep_imgs = [
        p
        for p in imgs
        if f"/{AREA}/aerial/" in p and Path(p).parent.name in zone_suffixes
    ]
    keep_msks = [
        p
        for p in msks
        if f"/{AREA}/labels/" in p and Path(p).parent.name in zone_suffixes
    ]
    assert len(keep_imgs) == len(keep_msks), "unpaired IMG/MSK after filter"
    return {"IMG": keep_imgs, "MSK": keep_msks}


d_train = zones_dict(imgs, msks, TRAIN_ZONES)
d_val = zones_dict(imgs, msks, VAL_ZONES)
print(f"Train zones {TRAIN_ZONES}: {len(d_train['IMG'])} tiles")
print(f"Val zone   {VAL_ZONES}: {len(d_val['IMG'])} tiles")

In [ ]:
dm = FlairDatamodule(
    data_dir=str(DATA_ROOT),
    dict_train=d_train,
    dict_val=d_val,
    batch_size=4,
    num_workers=0,
    use_metadata=False,  # no MTD files discovered for now
    norm_type="scaling",  # uint8 -> [0, 1] float
)
dm.setup("fit")

assert (
    dm.train_dataset is not None and dm.val_dataset is not None
)  # set by setup("fit")
print(f"Train dataset: {len(dm.train_dataset)} tiles")
print(f"Val dataset:   {len(dm.val_dataset)} tiles")
print(f"Channels:      {dm.channels}")
print(f"Norm type:     {dm.norm_type}")
print(f"num_classes:   {dm.num_classes} (start_one = {dm.start_one})")

loader = dm.train_dataloader()
batch = next(iter(loader))
img, msk = batch["img"], batch["msk"]
print(
    f"Batch shapes: img={tuple(img.shape)} ({img.dtype}), msk={tuple(msk.shape)} ({msk.dtype})"
)
print(f"img range after '{dm.norm_type}': [{img.min():.4f}, {img.max():.4f}]")
labels = msk.argmax(dim=1)  # one-hot back to class indices 0..12
print(
    f"Class indices present in this batch (0..12): {sorted(labels.unique().tolist())}"
)

## 3. Raw raster specs

In [ ]:
img_path, msk_path = d_train["IMG"][0], d_train["MSK"][0]
for label, path in [("IMG", img_path), ("MSK", msk_path)]:
    with rasterio.open(path) as src:
        print(f"--- {label}: {Path(path).name}")
        print(f"    size (WxH):  {src.width} x {src.height}")
        print(f"    count/dtype: {src.count} band(s), {src.dtypes}")
        print(f"    CRS:         {src.crs}")
        print(f"    resolution:  {src.res[0]:.2f} x {src.res[1]:.2f} m/px")
        print(f"    nodata:      {src.nodata}")
        print(f"    bounds:      {tuple(round(v, 1) for v in src.bounds)}")
    if label == "IMG":
        with rasterio.open(path) as src:
            data = src.read()
            for b, name in enumerate(BAND_NAMES, start=1):
                band = data[b - 1]
                print(
                    f"    band {b} ({name:}): min={band.min():3d} max={band.max():3d} mean={band.mean():7.2f}"
                )

## 4. Class distribution

Classes are heavily imbalanced — this is a stat to check early, and per-class
IoU (not just mIoU) will matter at evaluation time.

In [ ]:
N_SAMPLED = 80  # unless more exist in the sampled set

samples = d_train["MSK"] + d_val["MSK"]
random.shuffle(samples)
samples = samples[:N_SAMPLED]

counts = np.zeros(NUM_CLASSES, dtype=np.int64)
n_unmapped = 0
pixels = 0
for path in samples:
    with rasterio.open(path) as src:
        values = src.read(1).ravel()
    n_unmapped += np.count_nonzero((values < 1) | (values > NUM_CLASSES))
    values = values[(values >= 1) & (values <= NUM_CLASSES)]
    counts += np.bincount(values, minlength=NUM_CLASSES + 1)[1:]
    pixels += values.size

if n_unmapped:
    print(
        f"! {n_unmapped:,} pixels carry values outside 1..{NUM_CLASSES}; "
        f"read_msk() cannot one-hot them (they all-false) and argmax then maps "
        f"them to class 0 (Building) — flagging here."
    )
print(
    f"Sampled {len(samples)}/{len(d_train['MSK']) + len(d_val['MSK'])} masks "
    f"({pixels:,} mapped pixels)"
)
print(f"{'value':>5}  {'class':<22}  {'pixels':>14}  {'share %':>8}")
for value, cnt in enumerate(counts, start=1):
    share = 100 * cnt / pixels
    print(f"{value:>5}  {FLAIR_CLASSES[value]:<22}  {cnt:>14,}  {share:>8.3f}")

cmap = ListedColormap(CLASS_COLORS[:NUM_CLASSES])
_fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(
    [FLAIR_CLASSES[v] for v in range(1, NUM_CLASSES + 1)], counts, color=cmap.colors
)
ax.set_title(f"Pixel count per class — {AREA} ({len(samples)} masks sampled)")
ax.set_ylabel("pixels")
ax.set_yscale("log")
ax.tick_params(axis="x", rotation=45)
plt.setp(ax.get_xticklabels(), ha="right")
_fig.tight_layout()
plt.show()

## 5. Visualize tiles from the dataloader

Each row: true color, false color (NIR, Red, Green), elevation (nDSM),
labels grounded to class indices, and labels blended over true color.

In [ ]:
def plot_sample(img: torch.Tensor, msk: torch.Tensor, label: str = "") -> None:
    """Render one (C, H, W) image and its one-hot mask, straight from the loader."""
    rgb = img[[0, 1, 2]].permute(1, 2, 0).numpy().clip(0, 1)
    nir = img[[3, 0, 1]].permute(1, 2, 0).numpy().clip(0, 1)
    ndsm = img[4].numpy()
    labels = msk.argmax(dim=0).numpy()  # index i <-> MSK value i+1
    label_cmap = ListedColormap(CLASS_COLORS)

    _fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
    axes[0].imshow(rgb)
    axes[1].imshow(nir)
    axes[2].imshow(ndsm, cmap="gray")
    axes[3].imshow(labels, vmin=0, vmax=NUM_CLASSES - 1, cmap=label_cmap)
    axes[4].imshow(rgb)
    axes[4].imshow(labels, vmin=0, vmax=NUM_CLASSES - 1, alpha=0.45, cmap=label_cmap)
    for ax, name in zip(
        axes,
        [
            "true color (B1-3)",
            "false color (NIR,R,G)",
            "nDSM (band 5)",
            "labels",
            "labels over RGB",
        ],
    ):
        ax.set_title(f"{name} {label}", fontsize=9)
        ax.axis("off")
    plt.show()


patches = [
    Patch(facecolor=CLASS_COLORS[i], label=f"{i + 1}: {FLAIR_CLASSES[i + 1]}")
    for i in range(NUM_CLASSES)
]
_fig, ax = plt.subplots(figsize=(10, 1.4))
ax.legend(handles=patches, loc="center", ncols=4, frameon=False, fontsize=8)
ax.axis("off")
plt.show()

In [ ]:
for i in range(2):
    plot_sample(img[i], msk[i], label=f"(train[{i}])")

In [ ]:
batch_val = next(iter(dm.val_dataloader()))
for i in range(2):
    plot_sample(
        batch_val["img"][i],
        batch_val["msk"][i],
        label=f"(val tile {i}, zone {VAL_ZONES[0]})",
    )

## 6. takeaways & caveats

- **inventory**: flair-style `img_/msk_` geotiff pairs, 512×512 px, 0.2 m/px;
  imagery is 5-band uint8 (rgb + nir + ndsm), labels 1-band uint8 (1..13).
- **pairing is positional**: `discover_data()` sorts both lists independently;
  a zone with an unpaired tile would desynchronize every zone after it. A
  stem-equality check is cheap insurance (done in §1).
- **this notebook's train/val split is for illustration**: real splitting
  strategy (per-zone? per-area? domain-level?) is still todo — and per the
  official flair split, `d004_2021` is a *validation* domain, `d012_2019` a
  test domain; `d006_2020` is train. worth deciding how faithful we want to be
  before real training.
- **`num_classes=13`** is the flair 13-class nomenclature (nearest land-cover
  encoding available; our labels come from ocs ge rasterized onto these tiles
  — the mapping ocs ge → these 13 values should be documented in the pipeline
  that rasterized them).
- sentinel-2 `.npy` tiles are ignored by `discover_data()` for now.
- Nothing here hard-codes `/data/`; everything resolves from the repo root.